# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.17 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- **Lecture 01:** I used wc -l in Task 1.1 to count the lines in the CSV files and save the results to file_inventory.txt.
- **Lecture 02:** I created a function in Task 1.2 to count the rows, columns, and duplicate rows in each dataset.
- **Lecture 03:** In Task 4.3, my NumPy array had a shape of (27149, 2). I used axis=0 to calculate the mean across patients for each visit, which gave averages of 143.01 at baseline and 137.15 at week12.
- **Lecture 04:** I used pd.DataFrame() in Task 1.2 to organize the results of the five datasets into a table.
- **Lecture 05:** I standardized the gender values in Task 3.1 and used validation checks in Task 5.1 in order to make sure the cleaned data followed the rules.
- **Lecture 06:** I used pd.concat() to combine the north and south visit files in Task 2.2, then used merge() in Task 4.4 to join the cleaned datasets.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": table.duplicated().sum()
    }
results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))
table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**Intended rows and keys**
- patients.parquet: Each row represents one patient which are identified by patient_id.
- sites.csv: After I removed retired records, each row represented one current site, which are identified by site_code.
- visits_north.csv and visits_south.csv: Both files are stacked into one table. After I removed duplicates and kept the later reading for any rechecks, each row represented one visit for a patient. The combination of patient_id and visit identifies each row.
- a1c_wide.parquet: Each row originally represents one patient's A1c results, which are identified by patient_id. After I reshaped it to long format, each row represented one A1c result for a specific visit, which are identified by patient_id and visit.

**Issues with the raw data**
The patient and visit files contain exact duplicate rows and some visits also have repeated patient and visit combinations because of rechecks, so the later reading needs to be kept. The sites file contains current and retired records with the same site code, so only the current records should remain.

**Expected joins**
- Visits to patients: Many to one, because each patient can have multiple visits, but only one patient record.
- Patients to sites: Many to one, because multiple patients can belong to the same site.
- Visits to A1c results: One to one, because each patient and visit combination should have only one corresponding A1c result.

### 2.2 Stack the two visit files

In [4]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True)
south = south.rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)
visits_raw.shape

(57371, 7)

### 2.3 Audit the raw tables

In [5]:
patients = raw["patients"]
sites = raw["sites"]
visits = visits_raw
gender = patients["gender"].str.strip()
site_codes = patients["site_code"]
normalized_sites = site_codes.str.upper().str.replace("-", "").str.replace(" ", "")

sbp = pd.to_numeric(visits["sbp_mmhg"], errors="coerce")

dates = visits["visit_date"]
valid_dates = dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}") & pd.to_datetime(
    dates, format="%Y-%m-%d", errors="coerce"
).notna()

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "week 12": "week12",
    "wk12": "week12"
}

visit_keys = pd.DataFrame({
    "patient_id": visits["patient_id"],
    "visit": visits["visit"].str.strip().str.lower().map(visit_map)
})

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        ((gender != "") & ~patients["gender"].isin(["F", "M", "X"])).sum(),
        (gender == "").sum(),
        (patients["birth_year"] == "9999").sum(),
        (site_codes != normalized_sites).sum(),
        sites.duplicated(subset=["site_code"], keep=False).sum(),
        (~visits["visit"].isin(["baseline", "week12"])).sum(),
        (sbp == 999).sum(),
        ((sbp != 999) & ((sbp < 70) | (sbp > 250))).sum(),
        (~valid_dates).sum(),
        visit_keys.duplicated(subset=["patient_id", "visit"], keep=False).sum()
    ]
})

issue_audit.to_csv(AUDIT_PATH, index=False)
issue_audit

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
patients_clean = raw["patients"].copy(deep=True)
print(patients_clean["gender"].value_counts())
patients_clean = patients_clean.drop_duplicates()

gender_map = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nb": "X", "nonbinary": "X",
    "non-binary": "X"
}

patients_clean["gender"] = (
    patients_clean["gender"].str.strip().str.lower()
    .map(gender_map).astype("category")
)

patients_clean["birth_year"] = (
    pd.to_numeric(
        patients_clean["birth_year"].replace("9999", pd.NA),
        errors="coerce"
    ).astype("Int64")
)

patients_clean["site_code"] = (
    patients_clean["site_code"].str.upper()
    .str.replace("-", "")
    .str.replace(" ", "")
)

patients_clean = patients_clean.reset_index(drop=True)


gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]


In [7]:
patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04"
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum()
    ]
})

patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
patients_clean["gender"].cat.categories


Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates()
print(visits_clean["visit"].value_counts())

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "week 12": "week12",
    "wk12": "week12"
}

visits_clean["visit"] = (
    visits_clean["visit"].str.strip().str.lower().map(visit_map)
)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"], errors="coerce"
)

dates = visits_clean["visit_date"]
valid_dates = dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}")

visits_clean["visit_date"] = pd.to_datetime(
    dates.where(valid_dates), format="%Y-%m-%d", errors="coerce"
)

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]


In [9]:
rechecks = visits_clean[
    visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)
]
display(rechecks)

visits_clean = visits_clean.sort_values("visit_time")
visits_clean = visits_clean.drop_duplicates(
    subset=["patient_id", "visit"], keep="last"
)
visits_clean = visits_clean.sort_values(["patient_id", "visit"])
visits_clean = visits_clean.reset_index(drop=True)

visits_clean = visits_clean[
    ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]
]

visits_clean["visit"].value_counts(dropna=False)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121,72.3,visits_north.csv
...,...,...,...,...,...,...,...
55282,P28051,week12,2026-04-16,09:35,126,92.7,visits_south.csv
55472,P28232,baseline,2026-01-12,10:00,153,61.0,visits_south.csv
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv


visit
baseline    29599
week12      27582
Name: count, dtype: int64

In [10]:
visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum()
    ]
})

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame([
    ["patients", "exact duplicate rows",
     "Removed exact duplicates.",
     "They contain the same information so there is no need to keep them."],

    ["patients", "gender spellings that are not a code",
     "Changed gender values to F, M, or X and left unknown values missing.",
     "This keeps the gender categories consistent."],

    ["patients", "site codes needing normalization",
     "Made site codes uppercase and removed spaces and hyphens.",
     "This makes sure the site codes follow the same format."],

    ["patients", "birth_year sentinel 9999",
     "Replaced 9999 with missing values.",
     "9999 can't be a birth year."],

    ["visits", "weight column named wt_kg in visits_south.csv",
     "Renamed wt_kg to weight_kg.",
     "Both visit files need the same column names to combine them."],

    ["visits", "sbp sentinel 999 and readings outside 70 through 250",
     "Replaced these readings with missing values.",
     "These values aren't within the accepted blood pressure range."],

    ["visits", "impossible and loosely written visit dates",
     "Kept valid dates in YYYY-MM-DD format and made the others missing.",
     "This keeps the dates consistent and removes invalid dates."],

    ["visits", "repeated patient_id and visit pairs",
     "Kept the later reading for each repeated visit.",
     "Each patient should only have one record for each visit."],

    ["sites", "repeated site_code rows",
     "Keep only current site records.",
     "Retired records should be excluded so each site has one record."]
], columns=["table", "issue", "decision", "reason"])

decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table


,table,issue,decision,reason
0,patients,exact duplicate rows,Removed exact duplicates.,They contain the same information so there is ...
1,patients,gender spellings that are not a code,"Changed gender values to F, M, or X and left u...",This keeps the gender categories consistent.
2,patients,site codes needing normalization,Made site codes uppercase and removed spaces a...,This makes sure the site codes follow the same...
3,patients,birth_year sentinel 9999,Replaced 9999 with missing values.,9999 can't be a birth year.
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg.,Both visit files need the same column names to...
5,visits,sbp sentinel 999 and readings outside 70 throu...,Replaced these readings with missing values.,These values aren't within the accepted blood ...
6,visits,impossible and loosely written visit dates,Kept valid dates in YYYY-MM-DD format and made...,This keeps the dates consistent and removes in...
7,visits,repeated patient_id and visit pairs,Kept the later reading for each repeated visit.,Each patient should only have one record for e...
8,sites,repeated site_code rows,Keep only current site records.,Retired records should be excluded so each sit...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)

a1c_long["visit"] = a1c_long["visit"].map({
    "a1c_baseline": "baseline",
    "a1c_week12": "week12"
})
print(a1c_long["a1c_pct"].dtype)

a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        (a1c_long["visit"] == "baseline").sum(),
        (a1c_long["visit"] == "week12").sum()
    ],
    "a1c_pct_mean": [
        round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2),
        round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2)
    ]
})

a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary


float64


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]
try:
    patients_clean.merge(
        sites, on="site_code", how="left", validate="many_to_one"
    )
except pd.errors.MergeError as e:
    print(e)

display(sites[sites.duplicated(subset=["site_code"], keep=False)])

sites_current = sites.loc[
    sites["record_status"] == "current", ["site_code", "site_name"]
].copy()
sites_current


Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [14]:
merge_check = visits_clean.merge(
    a1c_long, on=["patient_id", "visit"], how="outer", indicator=True
)
print(merge_check["_merge"].value_counts())

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule"
    ],
    "count": [
        (merge_check["_merge"] == "both").sum(),
        (merge_check["_merge"] == "left_only").sum(),
        (merge_check["_merge"] == "right_only").sum(),
        (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
        (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
        (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
        len(sites) - len(sites_current)
    ]
})

merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64


,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [15]:
sbp_wide = visits_clean.pivot(
    index="patient_id",
    columns="visit",
    values="sbp_mmhg"
).reset_index()

sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].to_csv(
    SBP_WIDE_SAMPLE_PATH, index=False
)

sbp_wide.head()

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
1,P00002,132,129,-3
2,P00003,148,137,-11
3,P00004,142,<NA>,<NA>
4,P00005,139,127,-12


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
bp_array = np.array(complete_pairs, dtype=float)
print(bp_array.shape)
print(bp_array.mean(axis=0))

(27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [17]:
analysis = visits_clean.merge(
    patients_clean, on="patient_id", how="left", validate="many_to_one"
)

analysis = analysis.merge(
    sites_current, on="site_code", how="left", validate="many_to_one"
)

analysis = analysis.merge(
    a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one"
)

analysis = analysis[
    ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg",
     "a1c_pct", "gender", "birth_year", "site_code", "site_name", "source_file"]
]

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string"
})

analysis.shape

(57181, 11)

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
checks = pd.Series({
    "analysis row count matches visits": len(analysis) == len(visits_clean),
    "patient and visit pairs are unique": not analysis.duplicated(
        subset=["patient_id", "visit"]
    ).any(),
    "blood pressure values are valid": analysis["sbp_mmhg"].dropna().between(70, 250).all(),
    "site names are current": analysis["site_name"].dropna().isin(
        sites_current["site_name"]
    ).all(),
    "sbp_wide patient IDs are unique": sbp_wide["patient_id"].is_unique,
    "raw tables are unchanged": all(
        raw[name].equals(raw_snapshot[name]) for name in raw
    )
})

display(checks)
assert checks.all(), checks[~checks]

analysis row count matches visits     True
patient and visit pairs are unique    True
blood pressure values are valid       True
site names are current                True
sbp_wide patient IDs are unique       True
raw tables are unchanged              True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
analysis_sample = analysis[
    analysis["patient_id"].isin(SAMPLE_IDS)
].reset_index(drop=True)

analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)
analysis_sample

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [20]:
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string"
    },
    parse_dates=["visit_date"]
)

round_trip["visit_date"] = round_trip["visit_date"].astype("datetime64[us]")
print(round_trip.equals(analysis_sample))

True
